In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch

In [6]:
model_name = "Qwen/Qwen2.5-0.5B"

dataset = load_dataset(
    "text",
    data_files = "data/train.txt",
    split = "train"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [7]:
def tokenize_function(example):
    return tokenizer(example['text'])

tokenized_data = dataset.map(
    tokenize_function,
    batched = True,
    remove_columns=['text']
)

In [ ]:
print(tokenized_data)


Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 11
})


In [21]:
block_size = 32

def group_texts(examples):
    concatenated_examples = {}

    for key in examples.keys():
        concatenated_examples[key] = sum(examples[key], [])
    
    total_length = len(concatenated_examples['input_ids'])
    total_length = (total_length // block_size) * block_size # removing extra tokens

    result = {}

    for key, tokens in concatenated_examples.items():
        result[key] = [
            tokens[i:i + block_size] for i in range(0, total_length, block_size)
        ]

    result['labels']= result['input_ids'].copy()

    return result

In [22]:
lm_dataset = tokenized_data.map(
    group_texts,
    batched = True
)

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

In [26]:
print(len(lm_dataset)) #Dataset size

10


In [28]:
lm_dataset.set_format(
    "torch",
    columns = [
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

In [29]:
dataloader = DataLoader(
    lm_dataset,
    batch_size = 2,
    shuffle = True
)

In [31]:
for batch in dataloader:
    print(
        'Input_ids: ', 
        batch['input_ids'].shape
    )

    break

Input_ids:  torch.Size([2, 32])


In [33]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 5e-5
)

In [34]:
epochs = 3
model.train()   # model.train() doesn't mean "train the model." It means "set the model to training behavior." The actual learning happens through forward → loss → backward → optimizer.step().
for epoch in range(epochs): 
    for batch in dataloader: 
        print(f"Epoch {epoch+1}")
        outputs = model(
            input_ids = batch['input_ids'],
            attention_mask = batch['attention_mask'],
            labels = batch['labels']
        )

        loss = outputs.loss 
        print(f"loss: {loss.items()}")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()